## Basic Analog CiM Macro

Welcome to the CiMLoop Tutorials. This first tutorial will guide you through the
process of creating a basic analog CiM macro. Using this macro, we will explore
the CiMLoop syntax and important CiM components.

### Introduction and Overview

The diagram below shows the structure of the analog CiM macro we will be
creating. In the diagram, we show a 3x3 weight matrix and a 3x3 array for
illustrative purposes, but in simulation we'll be using a 32x32 array.

![Analog CiM Macro](images/how_cim_works.svg)

This analog CiM macro stores a weight matrix and performs matrix-vector
operations using analog computation. For simplicity, in this tutorial we assume
that analog components are high-enough resolution to process all operands.

The macro consists of three main components:

- Memory Array: A 2D array of memory elements that store the weight matrix. Each
  of these memory elements, which we call a CiM unit, stores a weight and
  performs an analog multiply-accumulate (MAC) operation between its stored
  weight and an analog input value.
- Row Drivers + DAC: These are the circuits that provide analog input values to
  the memory array. The Digital Analog Converter (DACs) convert digital input
  values to analog voltages. The row drivers then apply these voltages to the
  rows of the memory array. *In this tutorial, we'll be using a simple
  pulse-train DAC that is folded into the row drivers. We'll therefore only be
  seeing a row driver in the macro definition.*
- Column Drivers + ADC: These are the circuits that read out the analog output
  values from the memory array. Column drivers select and read of CiM array
  columns, and Analog Digital Converters (ADCs) convert analog outputs from each
  column into digital values.

We'll dive into the details of each of these components later on, but for now,
we'll start by looking at the structure and syntax of how to define this macro
in CiMLoop.

## Container-Hierarchy Representation

AccelForge uses a container-hierarchy representation to define the structure of CiM
macros. A **container** is a grouping of components and/or (sub)containers, and a
**container-hierarchy** is a series of containers where each contains all subsequent
components/containers. Expressing the CiM macro as a hierarchy lets us decompose the
macro into smaller, easier-to-understand components.

Using a container-hierarchy representation, we can break down our CiM array into three
levels.

### The Macro Level 

At the top level, we have the macro. The macro contains the ADC, column drivers, and row
drivers (note that the DAC is included in the row drivers).

Below the macro, we break the CiM array into column containers, where each container
represents one column of CiM units. Row drivers supply input values onto the rows; each
row connects to all columns, so these input values are *reused* between columns. Column
drivers read outputs from each column, and an ADC converts the analog outputs into
digital values.

Our three-column CiM array has three column containers.

### The Column Level

Looking inside each column, we see that there is no additional hardware; instead, the
column is immediately broken down further into multiple rows. Each row receives a
different input value, and the analog outputs are reused (*i.e.*, summed on a wire)
between multiple rows.

### The Row Level

Finally, looking inside each row, we see one `CimUnit` that stores a weight value. In
this macro, each `CimUnit` is capable of performing an analog MAC operation between an
8b input and an 8b weight. To represent this operation, we use 64x 1b MAC units, which
act as a virtual representation of the computation that occurs within the `CimUnit`.

In [ ]:
from _scripts import *

display_important_variables("basic_analog")
get_spec("basic_analog").arch

## AccelForge CiM Models

### Component and Container Definitions

In AccelForge, the architecture is a list of **components** (e.g., `Memory`, `Toll`,
`Compute`) and **containers** (e.g., `Container`). Each element has a name, and multiple
other attributes.

The YAML file for the `basic_analog` macro is shown below. It is annotated with comments
that describe each part.

We can identify several important parts of the file:

- **Top-level variables**: These are top-level configuration variables common across
  architectures. For fairness, they should be kept constant when comparing multiple
  architectures.
  - **Workload information**: The number of bits for each workload operand, the batch size,
    and histograms for each operand. The histograms represent the distribution of values
    that each operand can take on, and are used to calculate the energy of various
    components. Histograms listed here are just defaults, and will be overridden if
    more-specific histograms are provided for a given DNN layer.
  - **Microarchitecture**: The number of supported bits for each operand. These are usually
    used to set the sizes of buffers and other components.
  - **Circuits**: The technology node, supply voltage, and memory cell file are set here. We
    also define scaling factors for the energy and latency relative to the supply
    voltage.
  - *Calibration parameters*: These are used to calibrate the energy and area of various
    components to account for circuit-level differences and post-layout effects.

- **Arch variables**: These are variables that are used to configure this architecture
  specifically. They include:
  - **CiM array parameters**: The number of rows and columns in the array and the number of
    parallel inputs, outputs, and weights that are processed in each cycle. These are
    set automatically by the CiM processor upon processing a specification, and can be
    used in formulas to calculate other variables.
  - **Operand encoding**: The number of bits used to encode each operand and encoding
    functions. Note that, for each operand here, we define one encoding and one number
    of bits. In real systems, the encoding and number of bits may change as operands
    move through the system, so we may define additional encodings and/or numbers of
    bits in the variables file.
  - **Architecture & CiM Array Structure**: The number of memory cells of width and depth
    for the CiM unit is defined here. Cells along width store different bits of the same
    operand, computing a single MAC operation together. Cells along depth store
    different operands and are activated in different cycles. Finally, the bits per cell
    defines how many bits can be stored in a single memory cell.
  - **Data converters**: The ADC and DAC resolution are defined here, as well as the number
    of ADCs in a bank.
  - Other hardware parameters: The `cycle_period` of the system clock, is defined here,
    as well as the read pulse width, or the amount of time that a row is asserted during
    a read of the CiM array.
  - **Anything else**: Any other parameters that are specific to the macro being used can be
    defined here.

In [ ]:
txt = open(af.examples.arches.compute_in_memory.basic_analog).read()
display(
    Markdown(
        f"""
```yaml
{txt}
```
        """
    )
)

## Running the Accelerator

### Basic Run

To run the accelerator, we'll load in the spec using one of our pre-defined helper
functions get_spec. We can use the workload and renames from the `basic_analog`
example, which are written to fully utilize the CiM array.

The display_dnn_results function will display energy and latency of each Einsum, as well
as give energy and latency breakdowns.


We can make several observations from the results:

- The CiM array is fully utilized.
- The ADC and Row Drivers dominate the energy and latency of the macro.
- We can directly see the number of ADC and row driver (DAC) uses per MAC operation in
  the actions table at the bottom. Both of these come out to $1/32$ in this test, which
  reflects that the array is getting full reuse.

In [ ]:
basic_analog = af.examples.arches.compute_in_memory.basic_analog
workload = af.Workload.from_yaml(basic_analog, top_key="workload")
renames = af.Renames.from_yaml(basic_analog, top_key="renames")

spec = get_spec("basic_analog", add_dummy_main_memory=True, n_macros=1)
spec.workload = workload
spec.workload.einsums = spec.workload.einsums
spec.renames = renames
spec.mapper.metrics = af.Metrics.ENERGY

# It's OK to explore loop orders, but will make the search take longer, and is not very
# helpful for CiM accelerators. It's not very helpful because loop orders help us
# capture sliding window reuse, and the CiM macros consume very big tiles at once,
# giving limited benefit from the small overlap between each tile.
spec.mapper.explore_loop_orders = False

result = spec.map_workload_to_arch()
result = result.drop_components_with_zero_energy_and_latency()

display_dnn_results(result, title="Basic Analog Results")

We can visualilze the full mapping as a LoopTree by rendering the mapping result.

In [ ]:
result

Looking at the above, notice that it's fully utilizing the $32\times32$ CiM array by
mapping the $K$ and $N$ ranks across the rows and columns of the array, respectively.

### Exploring Array Sizes

Here, we'll evaluate how the energy efficiency and throughput of the macro
change with different array sizes. We'll run the macro while setting the
number of rows and columns to 16, 32, 64, and 128 and plot the throughput
and energy efficiency.


In [ ]:
import numpy as np


def run_basic_analog_spec(array_rows: int, array_columns: int):
    """Load basic_analog, resize the array, run the mapper, and return the result."""
    spec = get_spec("basic_analog", add_dummy_main_memory=True, n_macros=1)

    # Set the number of rows and columns in the array.
    spec.arch.find_spatial("row_ARRAY_ROWS").fanout = array_rows
    spec.arch.find_spatial("column_ARRAY_COLUMNS").fanout = array_columns

    # Match the workload to the array for a max-utilization test.
    spec.workload.einsums["Matmul"].rank_sizes["K"] = array_rows
    spec.workload.einsums["Matmul"].rank_sizes["N"] = array_columns

    spec.mapper.metrics = af.Metrics.ENERGY
    # It's OK to explore loop orders, but will make the search take longer, and
    # is not very helpful for CiM accelerators.
    spec.mapper.explore_loop_orders = False

    result = spec.map_workload_to_arch(print_progress=False)
    return result.drop_components_with_zero_energy_and_latency()


row_choices = [16, 32, 64, 128]
col_choices = [16, 32, 64, 128]

all_results = parallel(
    [
        delayed(run_basic_analog_spec)(array_rows=r, array_columns=c)
        for r in row_choices
        for c in col_choices
    ]
)


def _tops_per_w(r):
    return 2 / (float(r.energy()) / float(r.n_computes())) / 1e12


def _tops_per_mm2(r, rows, cols):
    area_m2 = sum(get_area_breakdown("basic_analog").values())
    spec = get_spec("basic_analog")
    spec.arch.find_spatial("row_ARRAY_ROWS").fanout = rows
    spec.arch.find_spatial("column_ARRAY_COLUMNS").fanout = cols
    spec.workload.einsums["Matmul"].rank_sizes["K"] = rows
    spec.workload.einsums["Matmul"].rank_sizes["N"] = cols
    evaluated = spec.calculate_component_costs()
    area_m2 = sum(float(v) for v in evaluated.arch.per_component_total_area.values())
    tops = 2 / (float(r.latency()) / float(r.n_computes())) / 1e12
    return tops / (area_m2 * 1e6) if area_m2 > 0 else 0.0


energy_efficiency_grid = np.zeros((len(row_choices), len(col_choices)))
throughput_grid = np.zeros((len(row_choices), len(col_choices)))
idx = 0
for i, rows in enumerate(row_choices):
    for j, cols in enumerate(col_choices):
        r = all_results[idx]
        idx += 1
        energy_efficiency_grid[i, j] = _tops_per_w(r)
        throughput_grid[i, j] = _tops_per_mm2(r, rows, cols)


fig, ax = plt.subplots(figsize=(10, 5), ncols=2)
for subplot, target, title in [
    (0, energy_efficiency_grid, "Energy Efficiency (TOPS/W)"),
    (1, throughput_grid, "Throughput (TOPS/mm^2)"),
]:
    im = ax[subplot].imshow(target, cmap="hot", aspect="auto")
    ax[subplot].set_title(title)
    ax[subplot].set_xlabel("Array Columns")
    ax[subplot].set_ylabel("Array Rows")
    ax[subplot].set_xticks(np.arange(len(col_choices)))
    ax[subplot].set_yticks(np.arange(len(row_choices)))
    ax[subplot].set_xticklabels(col_choices)
    ax[subplot].set_yticklabels(row_choices)
    ax[subplot].set_ylim(ax[subplot].get_ylim()[::-1])
    fig.colorbar(im, ax=ax[subplot])

plt.tight_layout()
plt.show()

To find out why the number of rows and columns influence energy efficiency,
we'll plot the energy breakdown of the CiM array as we vary the number of rows,
the number of columns, and then both at the same time.

We can see the following:

1. In all tests, the `cim_unit`s consume and the `row_drivers` consume little
   overall energy, while the `adc` dominates overall energy.
2. As the number of rows increases, `adc` energy per MAC decreases.
3. As the number of columns increases, `row_drivers` energy per MAC decreases.

These observations show the hints at one of the key design considerations in CiM
macros: how to reduce the area and energy cost of column readout circuitry, in
particular the ADCs. As we can see here, using more rows in the CiM array can
help reduce the energy cost of column readout circuits. This is just one of many
possible ways to reduce column readout energy. We'll explore other possibilities
later on.

Of course, this is just one macro; many other macros pay more energy for
`cim_unit` and `row_drivers` than for `adc`. There is a rich design
space to explore!

Before moving on, it may be helpful to stop for a moment and consider the
following questions:

1. Why does `adc` energy per compute decrease as the number of rows
   increase, but not the number of columns? And vice versa, why does
   `row_drivers` energy per compute decrease as the number of columns
   decreases, but not the number of rows?
2. Could there be other ways to decrease ADC, DAC, and row/column driver energy?

In [ ]:
def _run_basic_analog_array_size(rows, cols):
    spec = get_spec("basic_analog", add_dummy_main_memory=True, n_macros=1)
    spec.arch.find_spatial("row_ARRAY_ROWS").fanout = rows
    spec.arch.find_spatial("column_ARRAY_COLUMNS").fanout = cols
    spec.workload.einsums["Matmul"].rank_sizes["K"] = rows
    spec.workload.einsums["Matmul"].rank_sizes["N"] = cols
    result = spec.map_workload_to_arch(one_pbar_only=True)
    result = result.drop_components_with_zero_energy_and_latency()
    return rows, cols, result


row_choices = [32, 64, 128, 256]
col_choices = [32, 64, 128, 256]
results = parallel(
    [
        delayed(_run_basic_analog_array_size)(r, c)
        for r in row_choices
        for c in col_choices
    ]
)

In [ ]:
array_sizes = list(range(16, 257, 16))

results_vary_rows = parallel(
    [
        delayed(run_basic_analog_spec)(array_rows=s, array_columns=32)
        for s in array_sizes
    ]
)
results_vary_columns = parallel(
    [
        delayed(run_basic_analog_spec)(array_rows=32, array_columns=s)
        for s in array_sizes
    ]
)
results_vary_both = parallel(
    [delayed(run_basic_analog_spec)(array_rows=s, array_columns=s) for s in array_sizes]
)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for subplot, target, xlabel in [
    (0, results_vary_rows, "#Rows"),
    (1, results_vary_columns, "#Columns"),
    (2, results_vary_both, "#Rows and #Columns"),
]:
    breakdown = {
        a: {
            k: float(v) * 1e15
            for k, v in r.per_compute().energy(per_component=True).items()
            if v > 0
        }
        for a, r in zip(array_sizes, target)
    }
    bar_stacked(
        breakdown,
        title="",
        xlabel=xlabel,
        ylabel="Energy (fJ/MAC)",
        ax=ax[subplot],
    )
plt.tight_layout()
plt.show()

For more on how to reduce the energy of ADCs, we can look at the Titanium Law, which
states the following:

$$\frac{ADC\ Energy}{DNN}=\frac{Energy}{Convert} \times \frac{Converts}{MAC} \times \frac{MACs}{DNN} \times \frac{1}{Utilization}$$

Where:

- The ADC Energy per DNN is the amount of energy consumed by the ADCs to process
  a full DNN.
- The Energy per Convert is the amount of energy consumed by the ADCs to convert
  a single analog value to a digital value.
- The Converts per MAC is the number of analog-to-digital conversions required
  to perform a single multiply-accumulate operation.
- The MACs per DNN is the number of multiply-accumulate operations required to
  process a full DNN.
- The Utilization is the proportion of Converts per MAC that are actually used
  by the mapping.

The Titanium Law is introduced and discussed in the RAELLA paper
(https://dl.acm.org/doi/10.1145/3579371.3589062). Some questions in the lab will
require you to read RAELLA as well.

### Full-DNN Explorations

Here, we'll evaluate the energy of each component in the macro when running
the DNN ResNet-18. We'll report how the energy varies for each layer.

We'll use a larger array here, 64 columns by 512 rows, to ensure that we can
fit all weights.

In [ ]:
workload_path = af.examples.workloads.compute_in_memory.resnet18
workload = af.Workload.from_yaml(
    workload_path, top_key="workload", jinja_parse_data={"BATCH_SIZE": 1}
)
renames = af.Renames.from_yaml(
    workload_path, top_key="renames", jinja_parse_data={"BATCH_SIZE": 1}
)
spec = get_spec("basic_analog", add_dummy_main_memory=True, n_macros=1)
spec.workload = workload
spec.renames = renames

# Use a larger array (512 rows x 64 cols) to ensure we can fit all weights.
spec.arch.find_spatial("row_ARRAY_ROWS").fanout = 512
spec.arch.find_spatial("column_ARRAY_COLUMNS").fanout = 64
spec.workload = workload
spec.renames = renames
spec.mapper.metrics = af.Metrics.ENERGY
spec.mapper.explore_loop_orders = False
result = spec.map_workload_to_arch(one_pbar_only=True)
result = result.drop_components_with_zero_energy_and_latency()

fig, ax = plt.subplots(figsize=(20, 5))

energy = result.per_compute().energy(per_component=True, per_einsum=True)
for_plot = {}
for (einsum, component), v in energy.items():
    for_plot.setdefault(einsum, {})[component] = float(v) * 1e15

bar_stacked(
    for_plot,
    title="Per-Einsum Energy Breakdown",
    xlabel="Einsum",
    ylabel="Energy (fJ/MAC)",
    ax=ax,
)
plt.tight_layout()
plt.show()

per_einsum_energy = result.energy(per_einsum=True)
highest_energy_einsum = max(
    per_einsum_energy,
    key=lambda k: float(per_einsum_energy[k]),
)
print(f"Highest-energy Einsum: {highest_energy_einsum}")

Looking at the overall energy breakdown, we can see that the energy per MAC varies
greatly, with the `ADC` consuming substantially more energy for some layers rather than
others. To see why, we can look at the loop nest for some of the higher-energy layers.

The cell below shows the mapping for the workload. Take a spatial loop(s) associated
with the `row_ARRAY_ROWS` spatial fanout, which shows how the rows of the array are
being utilized. Take the product of the spatial loop bounds to find the number of rows
utilized. Are all 512 rows being utilized?

We know from the previous example that increasing the number of rows can decrease `ADC`
energy per MAC. If the workload does not have sufficient parallelism to utilize all
rows, however, this benefit will not be realized.

In [ ]:
result

# Editing Variables

This section builds off the basic CiM array tutorial above. Here we explore the
variables that parameterize the `basic_analog` macro and use them to run design
space explorations.

AccelForge stores two categories of variables in each architecture YAML:

- **Top-level variables** (the `variables:` block at the bottom of the YAML)
  contain values that should be matched between architectures when doing a fair
  comparison. These are typically the technology node, the supply voltage, the
  memory cell configuration, workload histograms, and various calibration
  scaling factors. In CiMLoop this is the `variables_iso.yaml` file.
- **Arch variables** (the `arch.variables:` block at the top of the YAML)
  contain design choices that are specific to this macro. These include the
  ADC/DAC resolution, the number of cells per CiM unit, the clock period, the
  encoding functions for inputs and weights, and any other knob the macro owner
  wants to expose. In CiMLoop this is the `variables_free.yaml` file.

Variables can refer to other variables using formulas. They are evaluated
in the order of top

We change variables by assigning to `spec.variables.<name>` (for top-level
variables) or `spec.arch.variables["<name>"]` (for arch-scoped variables). Note
that Pydantic will silently accept an assignment to the wrong namespace, so if
a variable change appears to have no effect, double-check which namespace it
belongs to.

The basic helper below wraps up these steps: grab the spec, set the array size,
apply the variable overrides, and run the mapper.


In [ ]:
def _run_basic_analog_variables(
    top_variables=None,
    arch_variables=None,
    tensor_bits=None,
    array_rows=32,
    array_cols=32,
    input_bits=8,
    weight_bits=8.0,
):
    """Load a basic_analog spec, apply variable overrides, and run the mapper."""
    spec = get_spec("basic_analog", add_dummy_main_memory=True, n_macros=1)

    spec.arch.find_spatial("row_ARRAY_ROWS").fanout = array_rows
    spec.arch.find_spatial("column_ARRAY_COLUMNS").fanout = array_cols

    # The min usage constraints can lock us in some weird suboptimal points when the
    # number of bits results in a weird max utilization point.
    spec.arch.find_spatial("row_ARRAY_ROWS").min_usage = 0
    spec.arch.find_spatial("column_ARRAY_COLUMNS").min_usage = 0

    spec.workload.einsums["Matmul"].rank_sizes["K"] = array_rows
    spec.workload.einsums["Matmul"].rank_sizes["N"] = array_cols

    if top_variables:
        for k, v in top_variables.items():
            setattr(spec.variables, k, v)

    if arch_variables:
        for k, v in arch_variables.items():
            spec.arch.variables[k] = v

    spec.workload.einsums["Matmul"].tensor_accesses["input"].bits_per_value = input_bits
    spec.workload.einsums["Matmul"].tensor_accesses[
        "weight"
    ].bits_per_value = weight_bits

    spec.mapper.metrics = af.Metrics.ENERGY_DELAY_PRODUCT
    spec.mapper.explore_loop_orders = False
    result = spec.map_workload_to_arch(print_progress=False)
    return result.drop_components_with_zero_energy_and_latency()

### Example 1: Voltage Scaling

First, we'll look at the effect of voltage scaling on the energy efficiency and
throughput of the macro. We'll sweep the voltage over a range of values and
plot the energy efficiency and throughput of the macro at each voltage.

The scaling factors defined in the top-level variables make energy scale
quadratically with voltage (`voltage_energy_scale = voltage ** 2`) and latency
scale linearly with voltage (`voltage_latency_scale = 1 / voltage`). We
therefore expect energy efficiency to drop and throughput to rise as voltage
increases.

In [ ]:
voltages = [v / 100 for v in range(55, 155, 5)]

voltage_results = parallel(
    [
        delayed(_run_basic_analog_variables)(top_variables={"voltage": v})
        for v in voltages
    ]
)

tops_per_w_by_v = {
    f"{v:.2f}V": 2 / r.per_compute().energy() / 1e12
    for v, r in zip(voltages, voltage_results)
}
tops_by_v = {
    f"{v:.2f}V": 2 / r.per_compute().latency() / 1e12
    for v, r in zip(voltages, voltage_results)
}

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(voltages, list(tops_per_w_by_v.values()), "o-")
ax[0].set_xlabel("Supply Voltage (V)")
ax[0].set_ylabel("Energy Efficiency (TOPS/W)")
ax[0].set_title("Supply Voltage vs. Energy Efficiency")
ax[0].grid(alpha=0.3)

ax[1].plot(voltages, list(tops_by_v.values()), "s-")
ax[1].set_xlabel("Supply Voltage (V)")
ax[1].set_ylabel("Throughput (TOPS)")
ax[1].set_title("Supply Voltage vs. Throughput")
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Example 2: Changing the Number of Input and Weight Bits

Next, we'll vary the number of bits for inputs and weights, seeing how this affects the
energy efficiency and throughput of the macro. We'll vary input and weight bits from 2
to 8 and plot the energy efficiency and throughput of the macro for each configuration.

To make things more interesting, we'll limit the DAC resolution and the bits per cell to
1. To process higher-resolution inputs and weights while using lower-resolution analog
components, we can *slice* the input and weight, or divide them into smaller pieces with
fewer bits, and process them separately. Different input slices will be processed in
different cycles, and different weight slices will be mapped to CiM units in the same
row and adjacent columns.

We can see that the architecture is set up to support slicing. We include the following
directives:

- `input_sliced`: This component operates on sliced inputs, and the number of actions
  scales with the number of input slices. We can see that the row drivers include this
  directive because they send one input slice at a time to the array.
- `weight_sliced`: This component operates on sliced weights, and the number of actions
  scales with the number of weight slices.
- `both_sliced`: This component operates both on sliced inputs and sliced weights. The
  ADC and column drivers also include this because they read an output for each weight
  slice (column) and input slice (timestep) combination.
- `both_sliced_weight_slices_parallelized`: This component operates on sliced inputs and
  sliced weights, and weight slices are parallelized across multiple components. The CiM
  unit includes this directive because it multiplies a sliced input by a sliced weight,
  and because weight slices are mapped across multiple CiM units.

Additionally, we can see that the column_ARRAY_COLUMNS fanout includes a directive
`usage_scale: n_weight_slices`. This is because weight slices are mapped across array
columns, so adding more weight slices proportionally increases the number of columns
required.

In [ ]:
bit_configs = [
    (input_bits, weight_bits)
    for input_bits in range(2, 9)
    for weight_bits in range(2, 9)
]

bit_results = parallel(
    [
        delayed(_run_basic_analog_variables)(
            input_bits=ib,
            weight_bits=wb,
            arch_variables={
                # Force one-bit slices
                "dac_resolution": 1,
                "bits_per_cell": 1,
            },
        )
        for ib, wb in bit_configs
    ]
)

tops_per_w_grid = {}
tops_grid = {}
for (ib, wb), r in zip(bit_configs, bit_results):
    tops_per_w_grid.setdefault(wb, {})[f"{ib} Input Bits"] = (
        2 / r.per_compute().energy() / 1e12
    )
    tops_grid.setdefault(wb, {})[f"{ib} Input Bits"] = (
        2 / r.per_compute().latency() / 1e12
    )

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

input_bits_list = list(range(2, 9))
weight_bits_list = list(range(2, 9))
for ib in input_bits_list:
    vals = [tops_per_w_grid[wb][f"{ib} Input Bits"] for wb in weight_bits_list]
    ax[0].plot(weight_bits_list, vals, "o-", label=f"{ib} Input Bits")
ax[0].set_xlabel("Number of Weight Bits")
ax[0].set_ylabel("Energy Efficiency (TOPS/W)")
ax[0].set_title("Number of Input and Weight Bits\nvs. Energy Efficiency")
ax[0].legend()
ax[0].grid(alpha=0.3)

for ib in input_bits_list:
    vals = [tops_grid[wb][f"{ib} Input Bits"] for wb in weight_bits_list]
    ax[1].plot(weight_bits_list, vals, "s-", label=f"{ib} Input Bits")
ax[1].set_xlabel("Number of Weight Bits")
ax[1].set_ylabel("Throughput (TOPS)")
ax[1].set_title("Number of Input and Weight Bits\nvs. Throughput")
ax[1].legend()
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

As shown in the plots, the energy efficiency and throughput both decrease as the number
of bits for inputs and weights increases. This is for two reasons.

1. Each additional input bit requires an additional cycle to process. The additional
   cycle increases latency and reduces throughput. Moreover, running the array for an
   additional cycle increases energy.
2. Each additional weight bit requires an additional CiM unit to store. This added
   hardware will reduce throughput and storage density, as the array can store and
   compute with fewer weights given the same number of CiM units. Moreover, performing
   fewer computations per array activation will increase the energy per computation.

### Example 3: Changing the Device Type

Finally, we'll look at the effect of changing the device type on the energy
breakdown of the macro. We'll compare two setups:

1. A macro that uses an **RRAM** device, storing four bits in each memory
   cell. Each CiM unit consists of one RRAM device.
2. A macro that uses an **SRAM** device, storing one bit in each memory cell.
   Each CiM unit consists of four SRAM devices to provide the same number of
   bits as the RRAM device.

To highlight the differences in devices, we'll use a larger array with 512
rows and 512 columns, providing 128 kB of storage in total.

In [ ]:
import os

SRAM_PATH = os.path.abspath(
    os.path.join(
        os.path.dirname(af.examples.arches.compute_in_memory.basic_analog),
        "memory_cells",
        "sram_example.yaml",
    )
)
RRAM_PATH = os.path.abspath(
    os.path.join(
        os.path.dirname(af.examples.arches.compute_in_memory.basic_analog),
        "memory_cells",
        "rram_example.yaml",
    )
)


def _run_device_scenario(cycle_seconds, title):
    """Run SRAM and RRAM scenarios for the given cycle time; plot the breakdown."""
    device_specs = [
        (
            "SRAM",
            {"cell_config": af.LiteralString(SRAM_PATH)},
            {
                "bits_per_cell": 1,
                "cim_unit_width_cells": 4,
                "cycle_period": cycle_seconds,
            },
        ),
        (
            "RRAM",
            {"cell_config": af.LiteralString(RRAM_PATH)},
            {
                "bits_per_cell": 4,
                "cim_unit_width_cells": 1,
                "cycle_period": cycle_seconds,
            },
        ),
    ]
    results = parallel(
        [
            delayed(_run_basic_analog_variables)(
                top_variables=tops,
                arch_variables=arch,
                array_rows=512,
                array_cols=512,
            )
            for _, tops, arch in device_specs
        ]
    )
    energies = {}
    for (label, _, _), r in zip(device_specs, results):
        breakdown = r.per_compute().energy(per_component=True)
        energies[label] = {k: float(v) * 1e15 for k, v in breakdown.items() if v > 0}

    fig, ax = plt.subplots(figsize=(8, 5))
    bar_stacked(
        energies,
        title=title,
        xlabel="Device Type",
        ylabel="Energy (fJ/MAC)",
        ax=ax,
    )
    plt.tight_layout()
    plt.show()
    return energies


display_markdown(
    """
First, we run a **fast** scenario, in which the macro is run with a 1e-7 s
(100 ns) cycle time. The energy breakdown for the fast scenario is shown below.
"""
)
fast_energy = _run_device_scenario(1e-7, "Fast Scenario (100 ns cycle)")

We can make several observations from this scenario:

1. For the CiM unit, RRAM energy is significantly higher than SRAM energy.
   This is because RRAM devices act as resistive elements, and current must
   be passed through them during a read, which consumes power. In contrast,
   SRAM cell reads do not require current to be passed through the cell and
   thus consume less read power.

2. For the column readout (ADC), energy is approximately equal between
   device types — the ADC runs at the column level and doesn't care about
   which cell technology backs the column.

3. The row drivers and CiM unit have complementary trade-offs that depend
   on the cell model. In the NeuroSim / hwcomponents RRAM model, driving
   analog voltage through the resistive devices shows up as row-driver
   energy. In the SRAM model, each cell stores only one bit, so four cells
   per CiM unit are used to match the 4b RRAM, but reads are
   sense-amplifier based and do not pass current through the cell. The
   exact split between `RowDrivers` and `CimUnit` depends strongly on the
   cell model; the total energy per MAC is the fairer comparison.

Next, we run a **slow** scenario, in which the macro is run at a 1e-4 s
(10 ms) cycle time. This simulates a scenario in which the macro must store
weights for a long time and is activated only intermittently.

In [ ]:
slow_energy = _run_device_scenario(100e-6, "Slow Scenario (100 us cycle)")

In this scenario, only one component of the energy breakdown changes. The SRAM cim_unit
energy is now much higher! This is because the SRAM devices require power to maintain
their state, and there is substantial leakage power in the SRAM devices over the 100us
cycle time. In contrast, the RRAM devices do not require power to maintain their state,
and thus do not leak power. The zero leakage of RRAM is one reason why it is an
attractive choice for low-power edge and intermittently-activated systems.

## Conclusion

Congratulations! You've completed the CiM tutorial. Now go explore all the CiM models
that AccelForge offers!